In [0]:
from pyspark.sql import functions as F

### Paths Definitions

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/"
checkpoint_path = "/Volumes/ecommerce/source_data/raw/_checkpoints/brz_order_items"
schema_location_path = "/Volumes/ecommerce/source_data/raw/_schemas/brz_order_items"

### Read Fact Stream using Auto Loader (cloudFiles)

In [0]:
df_fact_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location_path) # Auto Loader schema location
    .option("header", "true")
    .option("inferSchema", "true")
    .load(raw_data_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

### Write Fact Stream to Bronze Delta Table

In [0]:
(df_fact_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) # For Daily Batch Execution
    .toTable("ecommerce.bronze.brz_order_items")
)

In [0]:
%sql
-- SELECT COUNT(*) FROM ecommerce.bronze.brz_order_items;

COUNT(*)
183378


In [0]:
%sql

-- DROP TABLE ecommerce.bronze.brz_order_items;